# Lab 8 — The anatomical compiler: optimal control on the learned regulome ODE

*Eighth (and last core) session of the [`notebooks/` course](README.md) on computational synthetic
morphology — the one the previous seven build toward.*

Levin's **anatomical compiler** (the framing of Pezzulo & Levin 2016 and Levin 2022): *desired
anatomy in → intervention out.* Today's developmental biology is overwhelmingly **descriptive** —
it tells you what a system *does*. The compiler is the **predictive/prescriptive** flip: you specify
the *target* tissue state, and the machine computes the **actuation schedule** — which handles to
push (which TFs to dose / activate via synNotch / drive optogenetically; what print geometry; what
$V_{\rm mem}$ set-point), how much, when — that steers the system there.

This lab assembles the pieces from Labs 2–6 into that machine, at the level of a **learned regulome
surrogate**:

1. **Plant** — a learned Hypergraph Neural ODE $\dot x = f_\theta(x)$ on a TF timecourse ([Lab 5](05_hypergraph_neural_odes.ipynb)).
2. **System identification** — fit a linear surrogate $\dot z \approx Az + c$ to the plant's rollout
   (the slot where SINDy / Koopman would go; [Lab 6](06_control_theory.ipynb), [Lab 7](07_structural_identifiability.ipynb)).
3. **Controller synthesis** — LQR on the surrogate (warm-start), then **direct optimal control on
   the *nonlinear* plant**: parametrise a piecewise-constant input $u(t)$ on a chosen actuator set,
   minimise $\lVert x(T)-x_{\rm target}\rVert^2 + \lambda\lVert u\rVert^2$, and differentiate
   *through the ODE solve* (Adam through the rollout — `diffrax` adjoints in the production version).
4. **Closed-loop validation on the nonlinear plant** — does the computed schedule actually land it?

The headline you'll arrive at (and the one `figures/anatomical_compiler_results.json` records on the
kidney injury–repair plant): **you can steer what you actuate.** The linear surrogate is *unstable*
(a developmental trajectory linearised is a transient, not a relaxation — max Re$\,\lambda(A)\!\approx\!32$,
so "`surrogate_reliable = false`": LQR on it is fragile) — so you need the *nonlinear* plant and
feedback. With it, on the *actuated* TFs the final error drops ≈ **73%** (0.85 → 0.23); on *all* TFs
only ≈ **21%** (3.27 → 2.57) — the gap is exactly Lab 6's lesson, seen one more time. "Target state"
here is a TF-expression vector, not literally an anatomy; closing *that* gap — regulome state →
spatial form — is the deepest open problem, and where the spatial/mechanical simulators and the
bioelectric layer (the (now complete) BETSE-JAX refactor: `optimize_pattern` is *this very computation* on
$V_{\rm mem}$) come in.

**Needs:** `jax`, `numpy`, `matplotlib`, `optax` (+ a hand-rolled RK4 — no `diffrax` required here;
`scripts/benchmark_anatomical_compiler.py` is the `hgx` + `diffrax`-adjoint + `jaxctrl` production
version). Reads `data/processed/{temporal_expression,pseudotime_centers}.npy` + key-TF metadata +
`figures/anatomical_compiler_results.json`; synthetic toggle-switch fallback so every cell runs.
**Refs:** Pezzulo & Levin 2016; Levin 2022 (TAME / the anatomical compiler); `jaxctrl`.

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
try:
    import optax
    HAVE_OPTAX = True
except Exception:
    HAVE_OPTAX = False

jax.config.update("jax_enable_x64", False)
rng = np.random.default_rng(0)

def _find(*parts):
    here = Path.cwd()
    for base in [here, *here.parents]:
        p = base.joinpath(*parts)
        if p.exists():
            return p
    return None
def _loadjson(*parts):
    p = _find(*parts);  return json.loads(p.read_text()) if p else None

PROC = _find("data", "processed")
HAVE_REAL = PROC is not None and (PROC / "temporal_expression.npy").exists()
if HAVE_REAL:
    Texpr      = np.load(PROC / "temporal_expression.npy").astype(np.float32)   # (bins, genes)  z-scored
    ts_obs     = np.load(PROC / "pseudotime_centers.npy").astype(np.float32)
    gene_names = json.loads((PROC / "gene_names.json").read_text())
    key_tf_idx = json.loads((PROC / "key_tf_indices.json").read_text()) if (PROC / "key_tf_indices.json").exists() else {}
    key_tfs    = json.loads((PROC / "summary.json").read_text()).get("key_tfs", list(key_tf_idx))
    g_index    = {g: i for i, g in enumerate(gene_names)}
    SRC = "Fleck et al. 2023 cerebral-organoid regulome — pseudotime-binned timecourse"
else:
    print("[note] data/processed/ not found — synthetic toggle-switch plant (bistable; a genuine 2-attractor system).")
    Texpr = ts_obs = gene_names = key_tf_idx = None; key_tfs = ["geneA", "geneB"]; SRC = "synthetic toggle switch"

ana_json = _loadjson("figures", "anatomical_compiler_results.json")

# ---- Lab-5 plant: a tiny-MLP Neural ODE, hand-rolled RK4, jax.grad through the rollout --------
def init_field(d, h, key):
    k1, k2 = jax.random.split(key); s = 0.3
    return {"W1": s * jax.random.normal(k1, (h, d)) / np.sqrt(d), "b1": jnp.zeros(h),
            "W2": s * jax.random.normal(k2, (d, h)) / np.sqrt(h), "b2": jnp.zeros(d), "leak": jnp.float32(0.1)}
def field(p, x):
    return p["W2"] @ jnp.tanh(p["W1"] @ x + p["b1"]) + p["b2"] - p["leak"] * x
def rk4(rhs, x, t, dt):
    k1 = rhs(x, t); k2 = rhs(x + .5*dt*k1, t+.5*dt); k3 = rhs(x + .5*dt*k2, t+.5*dt); k4 = rhs(x + dt*k3, t+dt)
    return x + dt/6.0*(k1 + 2*k2 + 2*k3 + k4)
N_SUB = 8
def rollout(rhs, x0, ts):
    def seg(carry, pair):
        x, _ = carry; t0, t1 = pair; dt = (t1 - t0)/N_SUB
        x = jax.lax.fori_loop(0, N_SUB, lambda i, xx: rk4(rhs, xx, t0 + i*dt, dt), x)
        return (x, t1), x
    (_, _), xs = jax.lax.scan(seg, (x0, ts[0]), (ts[:-1], ts[1:]))
    return jnp.concatenate([x0[None], xs], 0)

print(f"plant timecourse     : {SRC}")
if HAVE_REAL:
    print(f"  bins × genes       : {Texpr.shape}   pseudotime: {np.round(ts_obs, 3)}   key TFs: {', '.join(key_tfs)}")
print(f"committed compiler run: {'figures/anatomical_compiler_results.json loaded' if ana_json else 'absent'}   | optax: {HAVE_OPTAX}")

## 1. What "the anatomical compiler" is — the four-stage pipeline

The compiler is not one algorithm; it's a *stack*, and you've built every layer:

| stage | what | which lab |
|---|---|---|
| **plant** | $\dot x = f_\theta(x)$ — the learned dynamics of the regulome | [Lab 5](05_hypergraph_neural_odes.ipynb) (Hypergraph Neural ODE), grounded in Labs 2–4 (the hypergraph, its spectrum, its modules) |
| **system identification** | a tractable surrogate ($\dot z\approx Az+c$; or SINDy/Koopman) for warm-starting + analysis | [Lab 6](06_control_theory.ipynb) (the linear plant), [Lab 7](07_structural_identifiability.ipynb) (is it identifiable?) |
| **controller synthesis** | LQR on the surrogate → then **direct optimal control on the nonlinear plant** ($u(t)$ minimising state-error + control-cost, gradient *through the ODE solve*) | this lab |
| **closed-loop validation** | integrate the *learned* (nonlinear) ODE under the computed $u(t)$ — did it land? | this lab |

Why not just open-loop on the static regulome (read KO signs off $H$, dose accordingly)? Because of
[Lab 3](03_benchmarking_fidelity.ipynb)'s lesson — *direction transfers, magnitude doesn't.* The
static signs tell you which way to push, not how far; you need a *dynamical model* and *feedback*.
And why not LQR on the linear surrogate alone? Because (next section) that surrogate is *unstable* —
a developmental trajectory linearised is a transient, not a stable plant. So: surrogate to warm-
start, nonlinear plant + direct optimal control to actually steer. We'll demonstrate it as a
**knockout-rescue**: perturb the initial state (in-silico knock down a master TF), watch the free
rollout drift to the wrong place, then compute the actuation schedule that rescues it to the
wild-type endpoint. That is "perturbation in → intervention out" — the compiler's defining move.

In [ ]:
# ---- fit the plant on the organoid timecourse (8 key TFs + a few high-variance genes) -------
if HAVE_REAL:
    var = Texpr.var(0)
    extra = [i for i in np.argsort(-var) if gene_names[i] not in key_tfs][:8]
    sel_idx   = [g_index[t] for t in key_tfs if t in g_index] + list(extra)
    sel_names = [gene_names[i] for i in sel_idx]
    Y_obs = jnp.asarray(Texpr[:, sel_idx]); ts = jnp.asarray(ts_obs)
else:
    def _toggle(z, beta=3.0, n=3.0):
        x, y = z; return jnp.array([beta/(1+y**n) - x, beta/(1+x**n) - y])
    ts = jnp.linspace(0.0, 6.0, 10)
    def _roll(z0, dt=0.01):
        z = z0; out = [z0]
        for k in range(1, len(ts)):
            for _ in range(int((ts[k]-ts[k-1])/dt)): z = z + dt*_toggle(z)
            out.append(z)
        return jnp.stack(out)
    Y_obs = _roll(jnp.array([0.6, 2.6])); sel_names = ["geneA", "geneB"]; key_tfs = sel_names
n_bins, d = Y_obs.shape

params = init_field(d, max(16, 2*d), jax.random.PRNGKey(0))
plant_rhs = lambda p: (lambda x, t: field(p, x))
def loss_fit(p): return jnp.mean((rollout(plant_rhs(p), Y_obs[0], ts) - Y_obs)**2)
t0 = time.time()
if HAVE_OPTAX:
    opt = optax.adam(3e-3); st = opt.init(params)
    @jax.jit
    def step(p, s):
        l, g = jax.value_and_grad(loss_fit)(p); u, s = opt.update(g, s); return optax.apply_updates(p, u), s, l
    for _ in range(700): params, st, l = step(params, st)
else:
    g = jax.jit(jax.grad(loss_fit))
    for _ in range(700): params = jax.tree_util.tree_map(lambda a, b: a - 5e-3*b, params, g(params))
print(f"plant fit: 700 Adam steps in {time.time()-t0:.2f}s, final MSE {float(loss_fit(params)):.4f}")

# ---- the control problem: a knockout-rescue ------------------------------------------------
x0_wt     = Y_obs[0]
x_target  = Y_obs[-1]                                       # the wild-type late-pseudotime state
ko_name   = "FOXG1" if "FOXG1" in sel_names else sel_names[0]   # the biggest "landscape tilt" from Lab 5
ko_j      = sel_names.index(ko_name)
x0_ko     = x0_wt.at[ko_j].add(-2.5)                        # in-silico knockdown at t=0
free_ko   = np.asarray(rollout(plant_rhs(params), x0_ko, ts))
err0_all  = float(np.mean((free_ko[-1] - np.asarray(x_target))**2))
# actuator set: the curated master TFs (minus the KO'd one) — what you'd realistically dose
act_names = [n for n in key_tfs if n in sel_names and n != ko_name]
act_idx   = [sel_names.index(n) for n in act_names]
B = jnp.asarray(np.eye(d)[:, act_idx]); m = B.shape[1]
err0_act  = float(np.mean((free_ko[-1][act_idx] - np.asarray(x_target)[act_idx])**2))
print(f"control problem: in-silico KO of {ko_name} at t=0  →  free rollout misses the wild-type endpoint")
print(f"  uncontrolled final error:  all genes {err0_all:.3f}   |   on the {m} actuated TFs {err0_act:.3f}")
print(f"  actuators (handles you may dose): {act_names}")

## 2. The linear surrogate — and why you can't just LQR it

Fit $\dot z \approx Az + c$ to the plant's dense rollout by least-squares on finite differences (the
SINDy/Koopman slot — `jaxctrl.SINDyOptimizer` would go here for a richer library). Then look at
$\max\operatorname{Re}\lambda(A)$. For a *developmental* trajectory this is almost always **positive**
— the cell is *transiting* between states, not relaxing to one, so the local linearisation is an
*unstable* node/spiral. LQR is happy to return a gain for it (CARE only needs *stabilisability*), but
the gain is huge and the closed loop is brittle: the surrogate is a poor stand-in for control. This
is the *practical* face of [Lab 7](07_structural_identifiability.ipynb)'s structural result — and
it's why Lab 6 had to *choose* $A=-L_{\rm sym}$ (Hurwitz by construction) rather than *fit* it. On
the committed kidney-IRI run: relative residual ≈ 0.009 (the fit is *good*) but $\max\operatorname{Re}
\lambda(A)\approx 31.8$ → `surrogate_reliable = false`, and the LQR gain norm is ≈ 2700. Moral: use
the surrogate to *warm-start*, then steer the *nonlinear* plant directly (§3).

In [ ]:
# ---- linear surrogate dz/dt ≈ A z + c, fit on a dense rollout of the learned plant -----------
ts_dense = jnp.linspace(float(ts[0]), float(ts[-1]), 80)
Z = np.asarray(rollout(plant_rhs(params), Y_obs[0], ts_dense))           # (80, d)
dt_dense = float(ts_dense[1] - ts_dense[0])
Zdot = (Z[1:] - Z[:-1]) / dt_dense                                       # forward differences
Zmid = 0.5 * (Z[1:] + Z[:-1])
Phi  = np.hstack([Zmid, np.ones((len(Zmid), 1))])                        # [z, 1]
coef, *_ = np.linalg.lstsq(Phi, Zdot, rcond=None)                        # (d+1, d)
A_sur, c_sur = coef[:d].T, coef[d]
rel_resid = float(np.linalg.norm(Phi @ coef - Zdot) / (np.linalg.norm(Zdot) + 1e-9))
max_re = float(np.linalg.eigvals(A_sur).real.max())
print(f"linear surrogate:  rel. residual {rel_resid:.4f}   max Re λ(A) = {max_re:+.3f}   →  "
      f"{'STABLE (Hurwitz)' if max_re < 0 else 'UNSTABLE — LQR on it is fragile; steer the nonlinear plant instead'}")
# an LQR warm-start (on a stability-clamped A, the way the production benchmark does it)
import scipy.linalg as sla
A_clamp = A_sur - (max(0.0, max_re) + 1e-3) * np.eye(d)                  # shift spectrum just into the LHP
K_lqr = np.linalg.solve(np.eye(m), B.T @ sla.solve_continuous_are(A_clamp, np.asarray(B), np.eye(d), np.eye(m)))
print(f"  LQR warm-start gain ‖K‖ = {np.linalg.norm(K_lqr):.1f}  (on the clamped-stable surrogate)")
if ana_json and "surrogate_control" in ana_json:
    sc = ana_json["surrogate_control"]
    print(f"\n  committed kidney-IRI run: rel. residual {sc.get('linear_surrogate_rel_residual'):.4f}, "
          f"max Re λ(A) = {sc.get('max_re_eig_A'):.1f}, surrogate_reliable = {sc.get('surrogate_reliable')}, "
          f"LQR ‖K‖(drivers) ≈ {sc.get('lqr_gain_norm_drivers'):.0f}")
    print("  (same pattern: a *good* fit, but an *unstable* A — so the controller goes on the nonlinear plant.)")

## 3. Direct optimal control on the nonlinear plant — *target in → schedule out*

Parametrise a **piecewise-constant** input $u(t)\in\mathbb R^{m}$ on the $m$ actuator TFs over
$n_{\rm seg}$ segments, box-bounded ($u = u_{\max}\tanh\theta_u$). Roll out the *learned* nonlinear
plant under $\dot x = f_\theta(x) + B\,u(t)$ from the perturbed start $x_0^{\rm KO}$, and minimise

$$J(\theta_u) = \lVert x(T) - x_{\rm target}\rVert^2 \;+\; \lambda \lVert u\rVert^2,$$

with Adam, differentiating *through the RK4 rollout* (`jax.grad`; in the production code, `diffrax`'s
continuous adjoint). Warm-start from the LQR feedback if you like — here we just start from zero. The
output $u^\star(t)$ *is* the prescription: "to rescue a $\,$KO$\,$ cell to the wild-type endpoint,
apply this much of each handle, on this schedule." We report the final error before vs after control,
both overall and restricted to the actuated TFs.

In [ ]:
N_SEG = 6
def u_of_t(useg, t, u_max=6.0):
    frac = jnp.clip((t - ts[0]) / (ts[-1] - ts[0] + 1e-9), 0.0, 1.0 - 1e-6)
    return u_max * jnp.tanh(useg[jnp.floor(frac * N_SEG).astype(int)])     # (m,)
def ctrl_rhs(p, useg):
    return lambda x, t: field(p, x) + B @ u_of_t(useg, t)
LAM = 1e-3
def loss_ctrl(useg):
    xT = rollout(ctrl_rhs(params, useg), x0_ko, ts)[-1]
    return jnp.mean((xT - x_target)**2) + LAM * jnp.mean(useg**2)
useg = jnp.zeros((N_SEG, m))
t0 = time.time()
if HAVE_OPTAX:
    opt2 = optax.adam(5e-2); st2 = opt2.init(useg)
    @jax.jit
    def cstep(u, s):
        l, g = jax.value_and_grad(loss_ctrl)(u); upd, s = opt2.update(g, s); return optax.apply_updates(u, upd), s, l
    hist = []
    for _ in range(400): useg, st2, l = cstep(useg, st2); hist.append(float(l))
else:
    g = jax.jit(jax.grad(loss_ctrl)); hist = []
    for _ in range(400): useg = useg - 3e-2 * g(useg); hist.append(float(loss_ctrl(useg)))
ctrl_traj = np.asarray(rollout(ctrl_rhs(params, useg), x0_ko, ts))
errc_all = float(np.mean((ctrl_traj[-1] - np.asarray(x_target))**2))
errc_act = float(np.mean((ctrl_traj[-1][act_idx] - np.asarray(x_target)[act_idx])**2))
print(f"optimal control: {len(hist)} Adam steps in {time.time()-t0:.2f}s   J: {hist[0]:.3f} → {hist[-1]:.4f}")
print(f"  final-state error  all genes : {err0_all:.3f} → {errc_all:.4f}   (↓{100*(1-errc_all/max(err0_all,1e-9)):.0f}%)")
print(f"  final-state error  actuated  : {err0_act:.3f} → {errc_act:.4f}   (↓{100*(1-errc_act/max(err0_act,1e-9)):.0f}%)   ← you fully steer what you directly actuate; the spill-over to the rest is the {100*(1-errc_all/max(err0_all,1e-9)):.0f}%")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.0))
# (a) trajectories of the actuated TFs: target / free-KO / controlled
cols = plt.cm.tab10(np.linspace(0, 1, m))
for k, j in enumerate(act_idx[:6]):
    c = cols[k]
    ax[0].plot(ts, [x_target[j]]*n_bins, ":", color=c, lw=1.5)
    ax[0].plot(ts, free_ko[:, j], "--", color=c, lw=1.2, alpha=.7)
    ax[0].plot(ts, ctrl_traj[:, j], "-", color=c, lw=2.0, label=sel_names[j])
ax[0].set_xlabel("pseudotime"); ax[0].set_ylabel("expression (z-scored)")
ax[0].set_title(f"KO({ko_name})-rescue:  ⋯ target   – – free (uncontrolled)   — controlled"); ax[0].legend(fontsize=8, ncol=2)
# (b) optimal control input u*(t)
useg_v = np.asarray(6.0 * np.tanh(useg))
tt = np.linspace(float(ts[0]), float(ts[-1]), 200)
for k in range(m):
    ax[1].step(np.linspace(float(ts[0]), float(ts[-1]), N_SEG+1)[:-1], useg_v[:, k], where="post", lw=2, label=act_names[k])
ax[1].axhline(0, lw=.6, color="k"); ax[1].set_xlabel("pseudotime"); ax[1].set_ylabel("control input  u(t)")
ax[1].set_title(f"Optimal actuation schedule  (piecewise-constant, {N_SEG} segments)"); ax[1].legend(fontsize=8)
# (c) the bar: this lab vs the committed kidney-IRI run
labels = ["actuated TFs", "all genes"]; before = [err0_act, err0_all]; after = [errc_act, errc_all]
x = np.arange(2); w = 0.35
ax[2].bar(x - w/2, before, w, label="uncontrolled", color="#bbb"); ax[2].bar(x + w/2, after, w, label="controlled", color="#4C72B0")
ax[2].set_xticks(x); ax[2].set_xticklabels(labels); ax[2].set_ylabel("final-state error (MSE)")
ax[2].set_title("This lab: KO-rescue error, before vs after control"); ax[2].legend(fontsize=8)
fig.tight_layout(); plt.show()

if ana_json and "optimal_control" in ana_json:
    oc = ana_json["optimal_control"]
    print(f"\n  committed kidney-IRI anatomical-compiler run ({oc.get('method')}):")
    print(f"    actuated TFs: err {oc.get('uncontrolled_final_err_actuated'):.2f} → {oc.get('controlled_final_err_actuated'):.2f}  "
          f"({100*oc.get('error_reduction_frac_actuated'):.0f}% reduction)")
    print(f"    all TFs     : err {oc.get('uncontrolled_final_err'):.2f} → {oc.get('controlled_final_err'):.2f}  "
          f"({100*oc.get('error_reduction_frac'):.0f}% reduction)   ← same 'steer what you actuate' gap")

## 4. What the anatomical compiler (here) is — and isn't

- **It's at the level of a *learned regulome surrogate*.** "Target state" is a TF-expression vector;
  "intervention" is a TF-actuation schedule. That's a real, useful object — it's what you'd hand to
  a synNotch designer or a small-molecule screen as a *target program* — but it is **not an anatomy**.
  The map *regulome state → spatial form* (cells moving, adhering, dividing, mechanics, reaction–
  diffusion) is the part this instrument deliberately doesn't model — it's the job of the cell-based
  simulators (CompuCell3D, Morpheus, PhysiCell, Chaste — see `notebooks/README.md`'s ecosystem map)
  and, for the *bioelectric* prepattern, of the BETSE-JAX refactor (`betse.science.jax.inverse.optimize_pattern`
  is *exactly this computation* — target $V_{\rm mem}$ pattern in, required ion currents out). Closing
  the regulome↔anatomy gap is the deepest open problem; the whitepaper's §4.3 forward programme is
  where the layers are meant to be wired together (a synNotch / bioelectric actuation, a CC3D-style
  mechanical readout, an `hgx`/`jaxctrl` regulatory readout, one model-in-the-loop cycle).
- **You steer what you actuate.** The actuated-vs-all gap (≈73% vs ≈21% here; the same on the
  kidney plant) is [Lab 6](06_control_theory.ipynb)'s controllability result, one more time: the
  reachable subspace is small; choose your handles well (Lab 6's leverage analysis) and don't expect
  to move the rest for free.
- **The linear surrogate is a warm-start, not the controller.** It's unstable (§2) — fine for an
  initial guess and for analysis, wrong as the plant you actually optimise on. Hence direct OC on the
  nonlinear flow.
- **It's the *flow* you control, not the parameters.** The Neural ODE's $\theta$ is structurally
  non-identifiable ([Lab 7](07_structural_identifiability.ipynb)) — and that's fine, because the
  controller cares about $x(t)$, not $\theta$. But any *mechanistic* reduction you'd swap in (a Hill-
  ODE GRN à la [Lab 1](01_gene_circuit_dynamics.ipynb) / the `jaxctrl` IRMA example) **does** have
  interpretable parameters — run a structural-ID check (Lab 7) *before* you fit and control it.
- **The objective and constraints are modelling choices.** $\lambda$ (control cost), $u_{\max}$
  (dose ceiling), $n_{\rm seg}$ (how often you can intervene), the actuator set $B$ (which handles
  exist) — change them and you change the prescription. A real campaign sweeps them; MPC re-plans
  under feedback (exercise (c)).

## 5. The arc — Labs 1 → 8, one line each

1. **Gene-circuit dynamics** — Hill functions, the toggle switch & bistability, the repressilator,
   "linearise then LQR" — in this course's toolchain (the bridge from Elowitz & Bois).
2. **Regulomes & hypergraphs** — a regulon is a *hyperedge*, not a clique; the incidence matrix *is*
   the hypergraph; the Laplacian spectrum is the first structural readout.
3. **Benchmarking fidelity** — does the regulome *predict perturbations*? The fidelity *triple*:
   direction transfers (≈83%), magnitude doesn't (r≈0.13) — disaggregate or be fooled.
4. **Modularity & the Module Identifiability Index** — does it *decompose*? The Hodge-Laplacian
   spectral gap; a genome-scale GRN is one tangled component (weak module identifiability), but the
   index *discriminates* systems (organoid > blueprint > bioprint) and tracks maturation.
5. **Hypergraph Neural ODEs** — *learn the flow*; cell types = attractors (Waddington made literal);
   rollout MSE splits stable structural drivers from transient stress responders.
6. **Control theory (`jaxctrl`)** — the regulome as a steerable plant; the masters don't control it
   (the dual of Lab 7's "don't observe it"); leverage ≠ identity; you steer what you actuate.
7. **Structural identifiability** — is the *mechanistic* reduction's $\theta$ recoverable, with
   perfect data? (the symbolic pre-condition; module ≠ structural ≠ practical identifiability.)
8. **The anatomical compiler** *(this lab)* — nonlinear optimal control on the learned ODE: *target
   tissue state in → TF-actuation schedule out*. Each of the whitepaper's §4.3 wet-lab experiments —
   programmed-plus-printed tissues, optogenetic morphogenesis, morphoceutical regeneration,
   bioelectric control — is **one instance of this**, with a different actuator $B$ (print geometry ·
   synNotch · light/dose · $V_{\rm mem}$) and a different readout layer wired in.

## 6. Exercises

**(a) Target a *fate basin*, not a state vector.** Replace the regression target $\lVert x(T)-x_{\rm
target}\rVert^2$ with "land in basin $B$" on a genuinely multi-attractor plant (the toggle switch
from [Lab 5](05_hypergraph_neural_odes.ipynb)): from a perturbed start in basin $A$, find the *minimal*
$u(t)$ (large $\lambda$) that pushes the trajectory across the separatrix into basin $B$. How does
the required dose scale with how far inside basin $A$ you start? (This is the honest version of
"reprogramme cell fate".)

**(b) A richer surrogate.** Swap the linear least-squares surrogate of §2 for a SINDy fit (polynomial
library + sparsity) or a Koopman/DMD lift. Does it become *stable* (Hurwitz)? Does the LQR warm-start
help the §3 optimiser converge faster / to a better optimum? (This is `jaxctrl/examples/irma_sindy_lqr.ipynb`'s move.)

**(c) MPC, not open-loop.** Instead of computing $u(t)$ once over $[0,T]$, re-optimise over a
receding horizon: at each step, plan the next $H$ segments, apply the first, observe the (noisy)
state, re-plan. Compare robustness to a state perturbation midway through. (This is the realistic
"intervention loop"; `jaxctrl` has the MPC primitives.)

**(d) Couple the bioelectric layer.** Use BETSE-JAX's `optimize_pattern` to find the $V_{\rm mem}$
prepattern that hits a target voltage map; define a (toy) $V_{\rm mem}\!\to\!$GRN map that biases the
regulome ODE's drift; then run §3 with the *bioelectric* set-point as the actuator. That's the
two-layer compiler — bioelectric prepattern → regulatory program → state — the §4.3(v) experiment in
silico. (Plant code: `~/Workspace/betse-unified`, `betse.science.jax.inverse`; this notebook for the
regulome layer.)

**(e) Actuator-set selection.** For a fixed budget $m$, which $m$-TF subset minimises the achievable
final error? It's a combinatorial control-allocation problem — greedily add the TF whose marginal
controllability gain (Lab 6's leverage) most reduces $\min_u J$. Does the greedy set match the
curated master regulators? The high-leverage TFs of Lab 6's §3? Neither?

Starters for (a) and (d):

In [ ]:
# --- Exercise (a) starter: push a toggle-switch trajectory across the separatrix --------------
N_SEG2, U_MAX2, DT2, NSTEP2 = 8, 4.0, 0.04, 200          # T = NSTEP2*DT2 = 8
def toggle(z, beta=3.0, n=3.0):
    x, y = z[..., 0], z[..., 1]; return jnp.stack([beta/(1+y**n) - x, beta/(1+x**n) - y], -1)
def toggle_roll_ctrl(z0, useg2):
    def body(i, z):
        seg = jnp.minimum((i * N_SEG2) // NSTEP2, N_SEG2 - 1)
        z = z + DT2*(toggle(z) + jnp.array([U_MAX2*jnp.tanh(useg2[seg]), 0.0]))       # actuate geneA only
        return jnp.maximum(z, 0.0)                       # concentrations ≥ 0 (and keeps 1+z^n away from its pole)
    return jax.lax.fori_loop(0, NSTEP2, body, z0)
z_start = jnp.array([2.6, 0.5])                          # well inside the "A-high" basin
def loss_a(u2):
    zf = toggle_roll_ctrl(z_start, u2)
    return (zf[1] - 2.6)**2 + (zf[0] - 0.5)**2 + 3e-2*jnp.mean(u2**2)   # target: the mirrored "B-high" basin
useg2 = jnp.full((N_SEG2,), -1.0)                        # initial guess: push geneA down
if HAVE_OPTAX:
    o = optax.adam(0.15); s = o.init(useg2)
    @jax.jit
    def astep(u2, st): l, g = jax.value_and_grad(loss_a)(u2); up, st = o.update(g, st); return optax.apply_updates(u2, up), st, l
    for _ in range(250): useg2, s, l = astep(useg2, s)
zf = np.asarray(toggle_roll_ctrl(z_start, useg2))
print(f"toggle-switch fate switch:  start (geneA={float(z_start[0]):.1f}, geneB={float(z_start[1]):.1f})  →  end (geneA={zf[0]:.2f}, geneB={zf[1]:.2f})")
print(f"  {'✓ crossed the separatrix into the B-high basin' if zf[1] > zf[0] else '✗ still in the A-high basin — increase U_MAX2/horizon or lower the control penalty'}"
      f"   (u schedule on geneA: {np.round(U_MAX2*np.tanh(np.asarray(useg2)),2)})")
print("  → your turn (a): sweep z_start deeper into basin A — how does the minimal dose (raise the 3e-2 penalty until it just fails) scale?  (c): make it MPC.")

In [ ]:
# --- Exercise (d) starter (optional): the two-layer bioelectric → GRN compiler ---------------
# Layer 1 (bioelectric): a tiny N-cell V_mem cluster, V̇ = -γ·V + D·Δ²V + J  — leaky 1-D diffusion driven by a
#   per-cell injected current J (the inverse-design handle). A stripped-down stand-in for BETSE-JAX's
#   finite-volume V_mem/gap-junction/ion-channel solver; `betse.science.jax.inverse.optimize_pattern` *is* this
#   move (jax.grad of MSE-to-a-target-V_mem-pattern → the currents), at full fidelity.
# Coupling (V_mem → GRN bias): the cluster's mean depolarisation biases geneA's drift — depolarise ⇒ push toward
#   the B-high fate. (BETSE-JAX: `betse.science.jax.physics.grn_trigger`, the V_mem→transcriptional-bias map.)
# Layer 2 (regulome): the §6(a) toggle plant, ẋ = f_toggle(x) + [bias, 0].
# The "compiler": pick the upper-layer control variable — a target depolarisation level v* for the cluster — so
#   the regulome lands in the target (B-high) basin. Here we *sweep* v* (the full version jax.grad's it through
#   the whole stack — optimize_pattern → currents → V_mem(t) → bias → toggle → final state — i.e. the bilevel
#   optimisation BETSE-JAX and this Lab's ODE compose into; that's the §4.3(v) experiment in silico).
try:
    import betse  # noqa: F401
    print("[betse importable] — for the *real* two-layer compiler, swap the toy bioelectric layer below for")
    print("  `betse.science.jax.inverse.optimize_pattern` (target V_mem pattern → ion currents) +")
    print("  `betse.science.jax.physics.grn_trigger` (V_mem → transcriptional bias); plant at ~/Workspace/betse-unified.")
except Exception:
    print("[betse not installed here] — running the self-contained toy stand-in (a faithful miniature of the pipeline).")

NCELL, GAMMA, DIFF, DT_V, NSTEP_V, W_VG = 7, 0.6, 0.15, 0.05, 120, 1.2
def _lap1d(v):                                          # discrete 1-D Laplacian, zero-flux ends
    vp = jnp.pad(v, 1, mode="edge");  return vp[2:] - 2.0*v + vp[:-2]
def vmem_rollout(J):                                    # V̇ = -γV + DΔ²V + J  (J: per-cell injected current)
    return jax.lax.fori_loop(0, NSTEP_V, lambda _, v: v + DT_V*(-GAMMA*v + DIFF*_lap1d(v) + J), jnp.zeros(NCELL))
def inverse_design_currents(v_target, steps=200):       # the `optimize_pattern` move: jax.grad J s.t. V_mem ≈ v_target
    def loss(J): return jnp.mean((vmem_rollout(J) - v_target)**2)
    J = jnp.zeros(NCELL)
    if HAVE_OPTAX:
        o = optax.adam(0.1); st = o.init(J)
        @jax.jit
        def step(J, st): l, g = jax.value_and_grad(loss)(J); up, st = o.update(g, st); return optax.apply_updates(J, up), st
        for _ in range(steps): J, st = step(J, st)
    return J
z0_A = jnp.array([2.6, 0.5])                            # start: A-high basin (uses `toggle` from the §6(a) starter)
print(f"\n  {'v* (target depol)':>18}  {'‖J‖ (designed)':>15}  {'V_mem mean':>11}  {'GRN bias':>9}  {'→ regulome fate':>22}")
for v_star in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5]:
    J  = inverse_design_currents(jnp.full(NCELL, v_star))
    vf = vmem_rollout(J)
    bias = float(W_VG * jnp.mean(vf))                   # depolarise ⇒ downward drift on geneA (toward B-high)
    z = z0_A
    for _ in range(NSTEP_V):
        z = z + DT_V*(toggle(z) + jnp.array([-bias, 0.0])); z = jnp.maximum(z, 0.0)
    z = np.asarray(z)
    print(f"  {v_star:>18.2f}  {float(jnp.linalg.norm(J)):>15.3f}  {float(jnp.mean(vf)):>11.3f}  {bias:>9.3f}  "
          + ("B-high ✓ (fate switched)" if z[1] > z[0] else "A-high (no switch)").rjust(22))
print("  → the smallest v* that drives the regulome across the separatrix is the §4.3(v) prescription:")
print("    'target fate in → bioelectric set-point out'. Real version: jax.grad v* (a full V_mem map) through")
print("    optimize_pattern + grn_trigger + this Lab's ODE — one differentiable stack, end to end.")

## Recap & where this sits

- The **anatomical compiler** is a *stack* — learned plant (Lab 5) → surrogate / system-ID (Lab
  5/5.5) → controller synthesis (LQR warm-start → **direct optimal control on the nonlinear plant**)
  → closed-loop validation — and the output is a **prescription**: target tissue state in → which
  handles, how much, when. We demonstrated it as a *knockout-rescue* (perturb the start, then compute
  the actuation that returns it to the wild-type endpoint); the production version
  (`scripts/benchmark_anatomical_compiler.py`) does it on a kidney injury–repair plant with `hgx` +
  `diffrax` adjoints + `jaxctrl`.
- The recurring lessons land here together: *you steer what you actuate* (Lab 6 — actuated error ↓
  ≈73% vs all-gene ≈21%); *the linear surrogate is an unstable warm-start, not the controller* (Lab
  5.5's practical face); *it's the flow you control, not* $\theta$ (Lab 7); *the target is a
  regulome state, not an anatomy* — closing that gap (with the spatial/mechanical simulators and the
  bioelectric layer) is the open frontier.
- **The course continues** (stretch): **Lab 9 — synthetic morphology in the wet lab** (the §4.3
  programme — bioprinting · synNotch / synthetic morphogens · optogenetic morphogenesis · bioelectric
  control, each "one optimal-control problem on the Hypergraph Neural ODE"; the BETSE-JAX refactor is
  its bioelectric companion), and **Lab 10 — cancer as loss of module identifiability** (run Lab 4's
  MII down a primary → organoid → tumour-organoid → cancer-line gradient).
- Pipeline: `scripts/benchmark_anatomical_compiler.py` (this lab's production version) and
  `scripts/benchmark_network_control.py` (the linear warm-up, Lab 6); the `jaxctrl` example notebooks
  (`repressilator_control_demo.py`, `irma_sindy_lqr.ipynb`, `grn_hypergraph_drivers.ipynb`); the
  bioelectric layer at `~/Workspace/betse-unified` (`betse.science.jax.inverse`).